# Unión de las bases geoespaciales — Malla Vial, Puentes, Malla Peatonal, Semáforos con los datos de la siniestralidad.

Este cuaderno **importa directamente las 4 capas ya procesadas y unidas** del cuaderno **Union_bases_malla_vial**:

- `Union_bases_malla_vial` → exporta `Data/processed/malla_urbana_unificada` 

> **Importante:** este notebook ya solo trabaja con esa union establecida

**Enfoque de unión:**

1. Se toma la **malla vial unificada** (salida de `Union_bases_malla_vial`) como capa base.
2. Cada **siniestro** se relaciona con el segmento vial más cercano mediante `sjoin_nearest`, igual que se hizo con puentes, semáforos y malla peatonal en el cuaderno de unión.

El cuaderno está organizado así:

1. Librerías
2. Malla vial unificada (capa base)
3. Datos de siniestralidad
4. Unión espacial de siniestros con la malla vial

## 1. Librerías

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 60)
plt.rcParams['figure.figsize'] = (9, 6)

c:\Users\Brayan\.conda\envs\proyecto\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\Brayan\.conda\envs\proyecto\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## 2. Malla vial unificada (capa base)

In [2]:
malla = gpd.read_file('Data/processed/malla_urbana_unificada.gpkg',layer="malla_vial")

### 2.1 Vista previa de los datos

In [3]:
malla

,vial_id,MVITIPO,MVISVIA,longitud_m,fragmento_id,longitud_seg_m,geometry
0,91428,CL,SD,30.958381,91428_1,30.958381,"LINESTRING (988757.414 997892.228, 988727.382 ..."
1,103844,CL,SD,35.729601,103844_2,35.729601,"LINESTRING (988786.272 997913.035, 988771.253 ..."
2,136273,DG,SD,31.900881,136273_3,31.900881,"LINESTRING (988935.627 997898.403, 988910.141 ..."
3,122955,KR,SD,48.556662,122955_4,48.556662,"LINESTRING (988745.967 997939.416, 988752.964 ..."
4,106448,DG,SD,21.075999,106448_5,21.075999,"LINESTRING (988910.141 997917.589, 988899.533 ..."
...,...,...,...,...,...,...,...
131700,94449,,SD,138.327740,94449_132005,138.327740,"LINESTRING (1002997.328 1025346.31, 1002881.52..."
131701,98523,,SD,47.720755,98523_132006,47.720755,"LINESTRING (1002881.529 1025421.974, 1002841.7..."
131702,107305,,SD,139.821854,107305_132007,131.444749,"LINESTRING (1002841.754 1025448.341, 1002730.8..."
131703,121584,,SD,351.231440,121584_132008,204.070575,"LINESTRING (1002841.754 1025448.341, 1002894.9..."


## 3. Datos de siniestralidad

In [4]:
siniestros = gpd.read_file("Data/processed/siniestros_procesados.gpkg")

if siniestros.crs.to_epsg() != 3116:
    siniestros = siniestros.to_crs(epsg=3116)

print(f"Registros de siniestros: {len(siniestros)}")
print(siniestros.crs)

Registros de siniestros: 23447
EPSG:3116


## 4. Unión espacial de siniestros con la malla vial

In [5]:
# Mismo patrón que puentes/semáforos/peatonal: sjoin_nearest asigna un único vial_id
# por siniestro (el más cercano), evitando que un evento cerca de una intersección
# se cuente para más de un segmento.
UMBRAL_SINIESTROS_M = 30  # más estricto que el 100m de las otras capas: aquí define el target, no contexto

siniestros_asignados = gpd.sjoin_nearest(
    siniestros, malla[["vial_id", "geometry"]],
    max_distance=UMBRAL_SINIESTROS_M, distance_col="dist_m"
)

# Reporte de cobertura, igual que en las capas anteriores — necesario para la sección
# de depuración/auditoría de tu tesis (antes/después por capa).
sin_match = siniestros["geometry"].size - siniestros_asignados["dist_m"].notna().sum()
print(f"Siniestros sin segmento asignado a {UMBRAL_SINIESTROS_M}m: {sin_match}")

siniestros_asignados = siniestros_asignados[siniestros_asignados["dist_m"].notna()].copy()

Siniestros sin segmento asignado a 30m: 1650
